# H2 - Objet auquel s'applique *fairness* selon le type d'acteurice

Question : l'objet auquel *fairness* s'applique depend-il du type d'acteurice ?

Pour chaque occurrence porteuse d'un objet (le noeud mot-cle a un enfant argument :ARGx ou un enfant :mod / :domain), on range l'objet dans la grille :

- technique : artefact technique (donnees, modele, algorithme, systeme...)
- procedure : decision, evaluation, conception, deploiement...
- social : personnes, groupes, traitement...
- mixte : l'objet releve d'au moins deux classes
- non classe : objet present mais hors lexique (a curer)

puis on croise avec le type d'acteurice. Les occurrences sans objet (invocations nues, enumerations, definitions a definiens-liste) sont exclues.

Memes conventions et meme avertissement que H1 : *fair-01* et *fairness* fusionnes, unite = occurrence, acteurice manquant exclu, jeu de test qui valide la chaine et non un resultat.

In [1]:
from pathlib import Path

DATA_DIR = Path(".")  # dossier contenant les donnees ; adapter a l'emplacement dans le depot
PENMAN_FILE = DATA_DIR / "fairness_sample_penmans.txt"
ACTOR_MAP   = DATA_DIR / "fairness_sample_doc_actor_map.csv"

# Concepts traites comme le mot-cle (fair-01 et fairness fusionnes)
KEYWORD_RE = r"^(fairness|fair-\d+)$"

N_PERMUTATIONS = 2000
RANDOM_SEED = 0

In [2]:
import re
import pandas as pd
import numpy as np
import penman
from scipy.stats import chi2_contingency

## Lexique objet -> grille

Petit lexique de depart, deliberement minimal : c'est la "mecanique parametree par mot-cle", l'inventaire partage des objets du discours. Le curer (decider ou vont *tool*, *product*, *intelligent-01*, *market*, *outcome*, etc.) est la tache editoriale principale de H2. La cellule finale liste les concepts non classes pour guider cette curation.

In [3]:
# Lexique editable. On garde un noyau defendable ; le reste tombe en "non classe".
GRID = {
    "technique": {"system","algorithm","data","datum","model","software",
                  "dataset","technology","machine","network"},
    "procedure": {"decide-01","decision","evaluate-01","evaluation","assess-01",
                  "process","process-01","design-01","develop-02","deploy-01",
                  "use-01","make-01","govern-01","governance","procedure"},
    "social":    {"person","people","individual","group","user","citizen",
                  "community","society","stakeholder","human","population",
                  "worker","treat-01"},
}
CONCEPT2GRID = {c: g for g, concepts in GRID.items() for c in concepts}
CAT_COL = "grid"

In [4]:
keyword_re = re.compile(KEYWORD_RE)
CORE_ARGS = {f":ARG{i}" for i in range(6)}   # arguments de coeur :ARG0..:ARG5

def parse_metadata(block):
    head = "\n".join(l for l in block.splitlines() if l.startswith("#"))
    sid = re.search(r"::id (\S+)", head)
    did = re.search(r"::doc_id (\S+)", head)
    snt = re.search(r"::snt (.+)", head)
    return (sid.group(1) if sid else None,
            int(did.group(1)) if did else None,
            snt.group(1).strip() if snt else "")

def canonical_edges(g):
    # remet les aretes inverses (:role-of) dans le sens tete -> dependant
    edges = []
    for s, r, t in g.edges():
        if r.endswith("-of") and r != ":consist-of":
            edges.append((t, ":" + r[1:-3], s))
        else:
            edges.append((s, r, t))
    return edges

In [5]:
def extract_objects(block):
    sid, did, snt = parse_metadata(block)
    graph_str = "\n".join(l for l in block.splitlines() if not l.startswith("#"))
    g = penman.decode(graph_str)
    concept = {v: c for v, _, c in g.instances()}
    E = canonical_edges(g)
    rows = []
    kw_vars = sorted(v for v, c in concept.items() if c and keyword_re.match(c))
    for i, v in enumerate(kw_vars):
        obj_concepts = [concept.get(dep) for h, r, dep in E
                        if h == v and (r in CORE_ARGS or r in (":mod", ":domain"))]
        if not obj_concepts:
            continue                      # occurrence sans objet -> hors H2
        classes = {CONCEPT2GRID.get(c) for c in obj_concepts}
        mapped = {c for c in classes if c is not None}
        if not mapped:
            grid = "non classe"
        elif len(mapped) == 1:
            grid = next(iter(mapped))
        else:
            grid = "mixte"
        rows.append(dict(sample_id=sid, doc_id=did, occ=i,
                         object_concepts="|".join(o or "?" for o in obj_concepts),
                         grid=grid, sentence=snt))
    return rows

In [6]:
raw = PENMAN_FILE.read_text(encoding="utf-8")
blocks = [b for b in raw.split("\n\n") if b.strip()]

obj_rows = []
for b in blocks:
    obj_rows += extract_objects(b)

df = pd.DataFrame(obj_rows)
print(f"occurrences porteuses d'un objet : {len(df)}")
print("\ndistribution dans la grille :")
print(df["grid"].value_counts().to_string())

occurrences porteuses d'un objet : 23

distribution dans la grille :
grid
non classe    17
technique      3
social         2
procedure      1


In [7]:
actor = pd.read_csv(ACTOR_MAP)[["doc_id", "actor_type", "needs_review"]]
df = df.merge(actor, on="doc_id", how="left")

n_nan = df["actor_type"].isna().sum()
print(f"occurrences sans acteurice (exclues du test) : {n_nan}")
print(f"documents acteurice marques needs_review : {int(actor['needs_review'].sum())} / {len(actor)} "
      "(etiquettes provisoires, non reverifiees)")

test_df = df.dropna(subset=["actor_type"]).copy()
print(f"occurrences retenues pour le test : {len(test_df)}")

occurrences sans acteurice (exclues du test) : 3
documents acteurice marques needs_review : 41 / 52 (etiquettes provisoires, non reverifiees)
occurrences retenues pour le test : 20


In [8]:
ct = pd.crosstab(test_df["grid"], test_df["actor_type"])
ct

actor_type,civil society,international organisation,national authority,private sector
grid,,,,
non classe,3,2,8,1
procedure,0,0,1,0
social,0,1,1,0
technique,0,0,3,0


## Test d'association

- chi2 d'independance : compare la table observee a celle qu'on aurait si la categorie et l'acteurice etaient sans lien. La p-value asymptotique n'est fiable que si chaque valeur attendue depasse ~5.
- comme la table est creuse, on ajoute un test de permutation : on melange les etiquettes d'acteurice N fois, on recalcule le chi2, et la p-value est la fraction des tirages au moins aussi extremes que l'observe. C'est l'equivalent robuste du test exact pour une table plus grande que 2 x 2.
- V de Cramer : force du lien, entre 0 (aucun) et 1 (parfait). Reperes conventionnels : 0,1 faible, 0,3 moyen, 0,5 fort.

On lit ensemble la p-value (le lien est-il reel ou du au hasard) et le V (est-il assez fort pour compter).

In [9]:
chi2, p_asym, dof, expected = chi2_contingency(ct)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))

# test de permutation (robuste pour cases creuses)
rng = np.random.default_rng(RANDOM_SEED)
labels_cat = test_df[CAT_COL].to_numpy()
labels_act = test_df["actor_type"].to_numpy()
count = 0
for _ in range(N_PERMUTATIONS):
    perm = rng.permutation(labels_act)
    c, _, _, _ = chi2_contingency(pd.crosstab(pd.Series(labels_cat), pd.Series(perm)))
    if c >= chi2:
        count += 1
p_perm = (count + 1) / (N_PERMUTATIONS + 1)

print(f"chi2 = {chi2:.2f}   dof = {dof}")
print(f"plus petite valeur attendue = {expected.min():.2f}  (si < 5, se fier a la permutation)")
print(f"p asymptotique (chi2)      = {p_asym:.3f}")
print(f"p permutation (B={N_PERMUTATIONS}) = {p_perm:.3f}")
print(f"V de Cramer                = {cramers_v:.3f}")

chi2 = 4.91   dof = 9
plus petite valeur attendue = 0.05  (si < 5, se fier a la permutation)
p asymptotique (chi2)      = 0.842
p permutation (B=2000) = 0.759
V de Cramer                = 0.286


## Concepts d'objet non classes (a arbitrer)

Ces concepts apparaissent comme objet mais ne sont pas dans le lexique. Ils indiquent ou etendre la grille.

In [10]:
unmapped = test_df.loc[test_df["grid"] == "non classe", "object_concepts"].value_counts()
print(unmapped.to_string() if len(unmapped) else "aucun")

object_concepts
tool              2
outcome           2
rank-01           1
observe-01        1
product           1
market            1
intelligent-01    1
thing             1
compete-01        1
algorithmic       1
distributive      1
learn-01          1


## Limites de cet echantillon

- Tres peu d'occurrences porteuses d'objet (la plupart des occurrences sont des invocations nues ou des enumerations), donc H2 est encore plus mince que H1 sur ce jeu de test.
- Une large part des objets tombe en "non classe" : le lexique de depart est minimal et doit etre cure ; c'est attendu et c'est l'information utile ici.
- Memes reserves que H1 : desequilibre entre acteurices, table creuse (se fier a la permutation), etiquettes d'acteurice provisoires, occurrences non independantes au sein d'un document.
- Le lexique GRID et la regle de l'objet double (categorie mixte) sont des decisions d'analyste a documenter.

En resume : la chaine d'extraction de l'objet et le test tournent de bout en bout ; le resultat n'est pas interpretable sur ce jeu de test et le lexique reste a construire.